# 07 - Step 15 (Novelty 1): Modern Vision Transfer Learning & Cross-Detector Few-Shot Adaptation
**L4 Group 23: Ismail Nooruddin, Munir Tariq, Ramsha Arshad**  
*CS/CE 351/363 - Artificial Intelligence, Fall 2026, Habib University*  
*Project Title: Classifying LIGO Glitches: Comparing CNNs and Simple Features*

---

### How to Run on Google Colab (Literally 3 Clicks!)
1. Go to **`Runtime → Change runtime type`** and select **`L4 GPU`** *(recommended with Google AI Pro — super fast & cheap on compute units!)* or **`T4 GPU`**.
2. Click **`Runtime → Run all`** (top menu).
3. Wait ~8–12 minutes. At the very end, your browser will automatically download **`step15_results.zip`** — drop that zip file straight into your local `AI Project/` folder!

### Scientific Objectives of Step 15
* **Part A (ImageNet → LIGO Transfer Learning):** Compare Glanzer et al.'s 4-layer CNN from scratch (`0.9466` Macro-F1) against three modern pretrained vision backbones:
  1. `ResNet-18 (1.0s Single-View)`
  2. `ResNet-18 (Multi-View RGB)` — maps `[0.5s, 1.0s, 2.0s]` zoom durations straight into `[R, G, B]` channels!
  3. `ConvNeXt-Tiny (Multi-View RGB)`
  4. `ViT-B/16 Vision Transformer (Multi-View RGB)`
* **Part B (Cross-Detector Few-Shot Domain Adaptation, `H1 ↔ L1` across 5 Seeds `[42–46]`):** Measure how much target-detector Macro-F1 recovers when a Hanford-trained model is gently fine-tuned on just **`5%`**, **`10%`**, or **`20%`** of Livingston glitches (and vice versa).


In [ ]:
# Step 15.1: Verify Colab GPU and Sync Latest Project Repository from GitHub
import os
import sys
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU      : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> L4 GPU or T4 GPU.")

REPO_URL = "https://github.com/IsmailNooruddin/ligo-glitch-classification.git"
REPO_DIR = "/content/ligo-glitch-classification"

if os.path.exists("src/models/transfer_models.py"):
    print(f"Already inside project repository: {os.getcwd()}")
else:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Verify Step 15 modules import cleanly
from src.models.transfer_models import TransferLIGOClassifier, SUPPORTED_BACKBONES
print("Step 15 modules verified. Supported backbones:", SUPPORTED_BACKBONES)


### Explanation of Cell 1: GPU Hardware Verification & Repository Sync
- **What this code does:** Checks that Colab attached a CUDA GPU (`L4` or `T4`), clones (or updates) your public GitHub repository `ligo-glitch-classification` into `/content/`, and verifies that `src/models/transfer_models.py` imports cleanly.
- **Why this matters:** Using GitHub as the single source of truth guarantees Colab runs the exact same modular code (`split_indices.json`, `metrics.py`, `dataset.py`) tested locally.


In [ ]:
# Step 15.2: Fast Fetch & Integrity Check for trainingsetv1d1.h5 (Zenodo Record 1486046)
import urllib.request
import json
import h5py

H5_PATH = "trainingsetv1d1.h5"

if os.path.exists(H5_PATH) and os.path.getsize(H5_PATH) > 2.5 * (1024 ** 3):
    print(f"Found valid local HDF5 dataset: {H5_PATH} ({os.path.getsize(H5_PATH)/(1024**3):.2f} GB)")
else:
    print("Fetching download URL from Zenodo API (record 1486046)...")
    download_url = "https://zenodo.org/records/1486046/files/trainingsetv1d1.h5?download=1"
    try:
        with urllib.request.urlopen("https://zenodo.org/api/records/1486046", timeout=15) as resp:
            rec = json.loads(resp.read().decode("utf-8"))
        for fobj in rec.get("files", []):
            if fobj.get("key") == "trainingsetv1d1.h5":
                download_url = fobj["links"]["self"]
                break
    except Exception as e:
        print(f"Zenodo API lookup fallback used ({e})")

    print(f"Downloading trainingsetv1d1.h5 from: {download_url}")
    !wget -q --show-progress -c -O {H5_PATH} "{download_url}"

# Verify HDF5 integrity before launching GPU training
with h5py.File(H5_PATH, "r") as f:
    root_classes = list(f.keys())
print(f"HDF5 Verified! Size={os.path.getsize(H5_PATH)/(1024**3):.2f} GB | Root Glitch Classes={len(root_classes)}")


### Explanation of Cell 2: Automated High-Speed Dataset Fetch & Verification
- **What this code does:** Checks if `trainingsetv1d1.h5` (~`2.84 GiB`) is already present; if not, downloads it directly inside Colab using Google's datacenter network (~15–40 seconds) from Zenodo record `1486046`, then opens the HDF5 file to verify all `22` root class groups are intact.
- **Why this matters:** You never have to wait half an hour uploading 3 GB from home Wi-Fi to Google Drive—everything runs reproducibly inside Colab.


In [ ]:
# Step 15.3: Run Complete Step 15 Pipeline (Part A Backbones + Part B 5-Seed Few-Shot Adaptation)
!python -u src/experiments/run_transfer_learning.py --h5-path trainingsetv1d1.h5 --output-dir outputs --zip-path step15_results.zip


### Explanation of Cell 3: Full GPU Execution (Part A + Part B)
- **What this code does:** Executes `src/experiments/run_transfer_learning.py` end-to-end on the GPU with CUDA mixed precision (`AMP`):
  1. **Part A (Pooled Seed `42`):** Trains and evaluates `ResNet-18 (1.0s)`, `ResNet-18 (Multi-View RGB)`, `ConvNeXt-Tiny (Multi-View RGB)`, and `ViT-B/16 (Multi-View RGB)` on the exact same `4,437 / 952 / 951` split used in Steps 7–12.
  2. **Part B (5-Seed Few-Shot Transfer `H1 ↔ L1`):** Across all `5` random seeds (`42–46`), measures `0% Zero-Shot`, `5%`, `10%`, `20%` target-site fine-tuning, and `100% Native` target upper bounds in both directions (`H1 → L1` and `L1 → H1`).
- **Outputs saved:** All 7 result CSVs, 3 high-res charts, the best fp16 checkpoint (`outputs/models/transfer_best_checkpoint.pt`), and `step15_results.zip`.


In [ ]:
# Step 15.4: Inspect Part A Pooled Transfer Learning Results & Visuals
from IPython.display import display, Image

pooled_df = pd.read_csv("outputs/tables/transfer_learning_pooled_results.csv")
per_class_df = pd.read_csv("outputs/tables/transfer_learning_per_class_f1.csv")

print("=" * 90)
print(" PART A: POOLED TRANSFER LEARNING RESULTS (UNSEEN TEST SET, N=951)")
print("=" * 90)
display(pooled_df)

print("\n=== PER-CLASS F1 COMPARISON ACROSS MODERN BACKBONES ===")
display(per_class_df)

print("\n--- Global Benchmark Leaderboard ---")
display(Image(filename="outputs/figures/transfer_learning_benchmark.png"))

print("\n--- Best Transfer Model Confusion Matrix ---")
display(Image(filename="outputs/figures/confusion_matrix_transfer_best.png"))


### Explanation of Cell 4: Part A Analysis — ImageNet Transfer & Multi-View RGB
- **What to inspect above:**
  - Compare `ResNet-18 (1.0s Single-View)` vs. `ResNet-18 (Multi-View RGB)` to isolate how much mapping `[0.5s, 1.0s, 2.0s]` into `[R, G, B]` helps temporal classes like `Repeating_Blips` and `Violin_Mode`.
  - Compare `ResNet-18` vs. `ConvNeXt-Tiny` vs. `ViT-B/16` against our Step 12 Adapted 4-Layer CNN (`0.9466` Macro-F1) and Step 9 Random Forest (`0.9376` Macro-F1).


In [ ]:
# Step 15.5: Inspect Part B Cross-Detector Few-Shot Adaptation (H1 <-> L1, 5 Seeds)
fs_summary = pd.read_csv("outputs/tables/few_shot_adaptation_summary.csv")
fs_per_class = pd.read_csv("outputs/tables/few_shot_per_class_recovery.csv")

print("=" * 90)
print(" PART B: CROSS-DETECTOR FEW-SHOT ADAPTATION (MEAN +/- STD ACROSS 5 SEEDS)")
print("=" * 90)
display(fs_summary)

print("\n=== KEY RECOVERY CLASSES (e.g. Violin_Mode, Repeating_Blips, No_Glitch) ===")
focus_classes = ["Violin_Mode", "Repeating_Blips", "No_Glitch", "Chirp", "Tomte", "Power_Line"]
display(fs_per_class[fs_per_class["Class"].isin(focus_classes)])

print("\n--- Few-Shot Cross-Detector Recovery Curves ---")
display(Image(filename="outputs/figures/few_shot_cross_detector_curve.png"))


### Explanation of Cell 5: Part B Analysis — Closing the Cross-Detector Domain Gap
- **What to inspect above:**
  - Look at the jump from `0% (Zero-Shot)` to `5%`, `10%`, and `20% Fine-Tuned` Macro-F1 in both directions (`H1 → L1` and `L1 → H1`).
  - Notice especially how quickly detector-shifted classes like `Violin_Mode` and `No_Glitch` recover once the model sees just a handful of target-observatory spectrograms!


In [ ]:
# Step 15.6: Download `step15_results.zip` Back to Your Laptop
import os

zip_file = "step15_results.zip"
if os.path.exists(zip_file):
    print(f"Ready to download: {zip_file} ({os.path.getsize(zip_file)/(1024**2):.2f} MB)")
    try:
        from google.colab import files
        files.download(zip_file)
    except ImportError:
        print("Not running in Google Colab UI — archive stays saved locally at:", os.path.abspath(zip_file))
else:
    print("ERROR: step15_results.zip was not found. Check Cell 3 logs above.")


### Explanation of Cell 6: Bringing Results Home to Local Repo
- **What this code does:** Triggers an automatic browser download of `step15_results.zip`, which bundles all 7 Step 15 CSV tables, 3 high-resolution PNG figures, and the best saved transfer checkpoint under clean `outputs/...` paths.
- **Next Action:** Drop `step15_results.zip` into your local `AI Project/` folder (you can also click **File → Save a copy in GitHub** in Colab if you want the executed cell outputs saved too) and tell your assistant **"done"** to verify everything and move to **Step 16 (VLMs / CLIP)**!
